# MLflow Experiment Tracking -- Track Every Experiment

Machine learning is inherently experimental. You tweak learning rates, swap model
architectures, adjust feature engineering -- and after a few days you have dozens of
runs with no record of what produced your best result.

**MLflow** solves this by giving every run a permanent, queryable record of:

| Concept | What it captures | Example |
|---|---|---|
| **Parameters** | Inputs you control | `learning_rate=0.001`, `n_estimators=200` |
| **Metrics** | Outputs you measure | `accuracy=0.95`, `val_loss=0.23` |
| **Artifacts** | Files you produce | `model.pkl`, `confusion_matrix.png` |
| **Tags** | Labels for organization | `model_type=random_forest`, `dataset=wine` |

This notebook walks through the `ExperimentTracker` class from
`agentexplorr.classical_ml.experiment_tracking`, which wraps MLflow into a clean,
educational interface.

> **Note**: You do not need a running MLflow server to follow along. The code cells
> below are structured as conceptual walkthroughs with inline comments.

In [ ]:
# --- Core imports ---
# In a real project you would also `import mlflow` directly, but our
# ExperimentTracker lazy-loads it so the import only happens when needed.

from agentexplorr.classical_ml.experiment_tracking import ExperimentTracker

# The five MLflow concepts you need to know:
#
# 1. EXPERIMENT  -- A named collection of runs (e.g., "wine_classification").
#                   Think of it as a project folder.
#
# 2. RUN         -- A single execution of your pipeline.  Every time you call
#                   `tracker.start_run(...)` a new run is created.
#
# 3. PARAMETERS  -- The knobs you set BEFORE training:
#                   learning_rate, batch_size, model architecture, etc.
#                   Logged once per run with `tracker.log_params(...)`.
#
# 4. METRICS     -- The numbers you MEASURE during or after training:
#                   accuracy, loss, F1 score, etc.
#                   Can be logged at each epoch (step) for time-series plots.
#
# 5. ARTIFACTS   -- Files produced by the run: model weights, plots, configs.
#                   Stored in MLflow's artifact store for later retrieval.

print("ExperimentTracker imported successfully.")
print("MLflow will be initialized lazily on first use.")

## Why Experiment Tracking Matters

Without tracking, ML development looks like this:

```
notebooks/
    model_v1.ipynb
    model_v2_final.ipynb
    model_v2_final_FINAL.ipynb
    model_v2_final_FINAL_actually_good.ipynb   <-- which one was best?
```

With MLflow, every run is automatically recorded in a structured database. The
**MLflow UI** (`mlflow ui` on the command line) lets you:

- Compare runs side-by-side (parameters, metrics, charts)
- Sort and filter by any metric (find the run with the highest accuracy)
- Download artifacts (retrieve the exact model weights from your best run)
- Reproduce results (every run records its git commit and parameters)

### Architecture

```
Your Code                     MLflow Tracking Server           Storage
---------                     ----------------------           -------
ExperimentTracker  -- HTTP --> REST API (port 5000)  -------> Backend store (SQLite/Postgres)
  .log_params()                                                 for params & metrics
  .log_metrics()                                      -------> Artifact store (local FS / S3)
  .log_artifact()                                               for model files & plots
```

The `ExperimentTracker` in `experiment_tracking.py` wraps all of this behind a
simple context-manager interface so you can focus on the ML, not the plumbing.

In [ ]:
# ============================================================================
# How to use ExperimentTracker -- a complete conceptual example
# ============================================================================
# NOTE: This cell shows the PATTERN you would use in a real project.
# It requires a running MLflow backend.  To run it yourself:
#   1. pip install mlflow
#   2. mlflow ui              (starts the server on http://localhost:5000)
#   3. Then execute this cell.
#
# The code below is wrapped in a try/except so the notebook does not fail
# if MLflow is not installed or no server is running.

try:
    # --- Step 1: Create a tracker for your experiment ---
    tracker = ExperimentTracker(
        experiment_name="notebook_demo",
        tracking_uri="./mlruns",       # local filesystem store (simplest option)
    )

    # --- Step 2: Start a run (context manager ensures it is closed properly) ---
    with tracker.start_run(
        run_name="random_forest_baseline",
        tags={"model_type": "sklearn", "dataset": "wine"},
        description="Baseline RF model with default hyperparameters",
    ) as run_id:
        print(f"Run started.  ID: {run_id}")

        # --- Step 3: Log parameters (inputs you set before training) ---
        tracker.log_params({
            "model": "RandomForestClassifier",
            "n_estimators": 100,
            "max_depth": 10,
            "criterion": "gini",
            "random_state": 42,
        })

        # --- Step 4: Log metrics (outputs you measure) ---
        # Final metrics (no step)
        tracker.log_metrics({
            "accuracy": 0.951,
            "f1_score": 0.943,
            "precision": 0.948,
            "recall": 0.939,
        })

        # Per-epoch metrics (use step= for time-series charts in the UI)
        for epoch, loss in enumerate([0.8, 0.5, 0.35, 0.28, 0.22]):
            tracker.log_metrics({"train_loss": loss}, step=epoch)

        # --- Step 5: Log artifacts (files produced by the run) ---
        # tracker.log_artifact("confusion_matrix.png")
        # tracker.log_model(trained_model, registered_model_name="wine_clf")

        # --- Step 6: Log complex data as JSON artifacts ---
        tracker.log_dict(
            {"feature_importances": {"alcohol": 0.32, "color_intensity": 0.21}},
            filename="feature_importances.json",
        )

    print("Run complete.  View results at: http://localhost:5000")

except Exception as e:
    # Gracefully handle missing MLflow installation
    print(f"MLflow demo skipped: {e}")
    print("\nTo run this demo yourself:")
    print("  1. pip install mlflow")
    print("  2. mlflow ui")
    print("  3. Re-run this cell")

## Model Registry and Deployment

Once you have a model you are happy with, MLflow's **Model Registry** provides
version control specifically designed for ML models:

```
Model Registry: "wine_classifier"
   Version 1  --  Accuracy 0.91  --  Stage: Archived
   Version 2  --  Accuracy 0.95  --  Stage: Production    <-- currently serving
   Version 3  --  Accuracy 0.93  --  Stage: Staging       <-- being validated
```

### Key operations in `ExperimentTracker`

| Method | What it does |
|---|---|
| `log_model(model)` | Serialize and store the trained model as an artifact |
| `log_model(model, registered_model_name="name")` | Store *and* register in the registry |
| `register_model(uri, name)` | Register an already-logged model by its artifact URI |
| `compare_runs(metric_key="accuracy")` | Rank all runs by a metric to find the best |
| `get_best_run(metric_key="accuracy")` | Shortcut to retrieve the single best run |

The `log_model` method in `experiment_tracking.py` auto-detects the framework
(scikit-learn or PyTorch) and uses the appropriate MLflow serializer, so you do not
need to worry about pickle vs `torch.save` -- just pass the model object.

### Deployment workflow

```
1. Train model  -->  2. log_model(registered_model_name="wine_clf")
                              |
                     3. Promote to "Production" in the UI or API
                              |
                     4. Load in production:
                        mlflow.pyfunc.load_model("models:/wine_clf/Production")
```

## Key Takeaways

1. **Every ML experiment should be tracked.** MLflow records parameters, metrics,
   artifacts, and source code so you can always reproduce and compare results.
2. **`ExperimentTracker`** in `experiment_tracking.py` provides a simplified wrapper
   around MLflow with lazy loading -- import it even if MLflow is not installed.
3. Use **`start_run()`** as a context manager to guarantee runs are properly closed,
   even if your training code raises an exception.
4. Log **parameters** once (before training) and **metrics** at each step or at the end.
   Use `step=epoch` for time-series visualization in the MLflow UI.
5. The **Model Registry** adds version control and stage management (Staging,
   Production, Archived) on top of artifact storage.
6. **`compare_runs()`** and **`get_best_run()`** let you programmatically find the
   best-performing configuration across all your experiments.

## Next Steps

- Start the MLflow UI locally: `mlflow ui` and open http://localhost:5000.
- Integrate tracking into a full training pipeline using the `Trainer` from
  `training_loop.py` -- log metrics at each epoch inside the training loop.
- Explore alternatives: [Weights & Biases](https://wandb.ai/) and
  [DVC](https://dvc.org/) offer similar capabilities with different trade-offs.